# Модуль A

## Пункт 1.1

### Загрузка библиотек

In [4]:
import pandas as pd
import numpy as np
import requests
import gpxpy
import os
import matplotlib.pyplot as plt
import osmnx as ox
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from tqdm import tqdm
import psycopg2
from psycopg2.extras import execute_values
from typing import List, Tuple
from PIL import Image
import io
from geopy.distance import geodesic
from shapely.geometry import Point
from concurrent.futures import ThreadPoolExecutor, as_completed

tqdm.pandas()

### Загрузка данных в систему

In [114]:
# Функция для загрузки данных
def download_file(url):
    local_filename = url.split('/')[-1]
    r = requests.get(url)
    f = open(local_filename, 'wb')
    # Загрузка по частям (для больших файлов)
    for chunk in r.iter_content(chunk_size=512 * 1024): 
        if chunk: 
            f.write(chunk)
    f.close()

### Преобразование данных в pandas DataFrame

In [5]:
# Функция для получения всех ссылок на файлы
def list_files(dir):
    r = []
    for root, dirs, files in os.walk(dir):
        for name in files:
            r.append(os.path.join(root, name))
    return r

In [6]:
base_dir = "workout-routes"
file_paths = list_files(base_dir)[:3] # Рассмотрим только первые 3 трека

In [118]:
Track_ids = [] # Идентификатор трека
DateTime = [] # Дата и время
Lat = [] # Широта точки
Lng = [] # Долгота точки
Alt = [] # Высота над уровнем моря

for track_id, file_path in enumerate(file_paths):
    gpx_file = open(file_path, "r")
    gpx = gpxpy.parse(gpx_file)
    for track in gpx.tracks:
        for segment in track.segments:
            points = segment.points
            N = len(points)
            for i in range(N):
                point = points[i]
                datetime = point.time
                lat = point.latitude
                lng = point.longitude
                alt = point.elevation
                Track_ids.append(track_id + 1)
                DateTime.append(datetime)
                Lat.append(lat)
                Lng.append(lng)
                Alt.append(alt)

In [119]:
data = pd.DataFrame({
    "Track_id": Track_ids,
    "DateTime": DateTime,
    "Latitude": Lat,
    "Longitude": Lng,
    "Altitude": Alt 
})

data

,Track_id,DateTime,Latitude,Longitude,Altitude
0,1,2022-03-22 17:47:32+00:00,28.526319,77.205764,233.448227
1,1,2022-03-22 17:47:33+00:00,28.526304,77.205761,233.366989
2,1,2022-03-22 17:47:34+00:00,28.526298,77.205760,233.334747
3,1,2022-03-22 17:47:35+00:00,28.526288,77.205757,233.280121
4,1,2022-03-22 17:47:36+00:00,28.526277,77.205755,233.228973
...,...,...,...,...,...
1682,3,2022-07-26 08:38:16+00:00,28.672763,77.249646,211.568863
1683,3,2022-07-26 08:38:17+00:00,28.672766,77.249640,211.554367
1684,3,2022-07-26 08:38:18+00:00,28.672769,77.249634,211.539078
1685,3,2022-07-26 08:38:19+00:00,28.672772,77.249628,211.523132


### Получение топографических карт с треками маршрутов

In [120]:
import osmnx as ox
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

def visualize_topo_map(points, center_lat, center_lng, radius=500, filename="map.png"):
    try:
        # Получаем данные
        G = ox.graph_from_point((center_lat, center_lng), dist=radius, network_type="all", simplify=True)
        
        # Инициализация фигур
        fig, ax = ox.plot_graph(G, show=False, close=False, 
                              edge_color="gray", node_size=0, 
                              bgcolor="white", figsize=(10, 8))
        

        # 1. Водоёмы (синий)
        try:
            water = ox.features_from_point((center_lat, center_lng), 
                                         tags={"natural": "water"}, 
                                         dist=radius)
            if not water.empty:
                water.plot(ax=ax, color="blue", alpha=0.5)
        except:
            pass

        # 2. Здания (оранжевый)
        try:
            buildings = ox.features_from_point((center_lat, center_lng), 
                                              tags={"building": True}, 
                                              dist=radius)
            if not buildings.empty:
                buildings.plot(ax=ax, color="orange", alpha=0.7)
        except:
            pass

        # 3. Зелёные зоны (зелёный)
        try:
            green = ox.features_from_point((center_lat, center_lng), 
                                        tags={"landuse": "grass", "natural": "wood"}, 
                                        dist=radius)
            if not green.empty:
                green.plot(ax=ax, color="green", alpha=0.3)
        except:
            pass

        # Трек маршрута
        lngs, lats = zip(*points)
        ax.plot(lngs, lats, color="red", linewidth=3)


        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close()
        return filename

    except Exception as e:
        print(f"Ошибка при создании карты: {e}")
        return filename

In [121]:
# Загрузка карт в папку maps
base_dir = "maps"

for track_id, file_path in tqdm(enumerate(file_paths)):
    # Чтение gpx формата
    gpx_file = open(file_path, "r")
    gpx = gpxpy.parse(gpx_file)

    # Широта и долгота точек
    Lat = []
    Lng = []
    for track in gpx.tracks:
        for segment in track.segments:
            points = segment.points
            N = len(points)
            for i in range(N):
                point = points[i]
                lat = point.latitude
                lng = point.longitude
                Lat.append(lat)
                Lng.append(lng)
    
    points=tuple(zip(Lng, Lat))
    center_lat = np.mean(Lat)
    center_lng = np.mean(lng)
    filename = os.path.join(base_dir, str(track_id + 1) + ".png")

    topo_map = visualize_topo_map(points, center_lat, center_lng, filename=filename)

3it [00:13,  4.57s/it]


### Загрузка в базу данных

In [7]:
def get_conn(dbname, user, password, host, port):
    return psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)

In [8]:
dbname = "Routes_tracks" # Название базы данных
user = "postgres" # Имя пользователя для подключения
password = "86754231qaZ" # Пароль пользователя для подключения
host = "localhost" # Хост
port = "5434" # Порт

conn = get_conn(dbname, user, password, host, port)

In [124]:
# Создание таблицы в базе данных
def create_table(query):
    try:
        cur = conn.cursor()
        cur.execute(query)
        conn.commit()

    except Exception as e:
        print("Ошибка:", e)
        conn.close()

In [125]:
# Добавление данных в таблицу бд без карты
def insert_data(query, data):
    try:
        cur = conn.cursor()
        execute_values(cur, query, data)
        conn.commit()

    except Exception as e:
        print("Ошибка:", e)
        conn.close()

In [126]:
# Добавление данных в таблицу бд с картой
def insert_data_with_image(query, data, image_path):
    try:
        cur = conn.cursor()
        with open(image_path, 'rb') as file:
            binary_data = file.read()
        cur.execute(query, (data, binary_data,))
        conn.commit()

    except Exception as e:
        print("Ошибка:", e)
        conn.close()

In [108]:
query = """CREATE TABLE IF NOT EXISTS Track_Maps (
    id SERIAL PRIMARY KEY,
    track_id INTEGER NOT NULL UNIQUE,
    image BYTEA NOT NULL
);"""

create_table(query)

In [109]:
query = """CREATE TABLE IF NOT EXISTS Track_Points (
    id SERIAL PRIMARY KEY,
    track_id INTEGER NOT NULL REFERENCES Track_Maps (track_id),
    \"DateTime\" TIMESTAMP NOT NULL,
    Latitude NUMERIC NOT NULL,
    Longitude NUMERIC NOT NULL,
    Altitude NUMERIC NOT NULL
);"""

create_table(query)

In [128]:
query = """INSERT INTO Track_Maps (track_id, image)
        VALUES (%s, %s)"""

for track_id in range(1, 4):
    image = os.path.join(base_dir, str(track_id) + ".png")
    insert_data_with_image(query, track_id, image)

In [129]:
query = """INSERT INTO Track_Points (track_id, \"DateTime\", Latitude, Longitude, Altitude)
        VALUES %s"""

insert_data(query, data.values.tolist())

## Пункт 1.2

### Получение окружающей среды для каждой точки трека

In [9]:
# Загружаем заново данные из базы данных
data = pd.read_sql("SELECT * FROM Track_Points", conn).drop(columns="id")

/var/folders/lq/13c4rlw92f7_q2025mqwt1fm0000gn/T/ipykernel_8039/4146111454.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql("SELECT * FROM Track_Points", conn).drop(columns="id")


In [15]:
def get_track_environment(points, radius=500):
    results = []
    
    for i, (track_id, lng, lat, alt) in tqdm(enumerate(points)):
        point_data = {
            'point_id': i,
            'track_id': track_id,
            'latitude': lat,
            'longitude': lng,
            'altitude': alt,
            'water_dist': -1,
            'building_dist': -1,
            'green_dist': -1,
            "water_area": -1,
            "building_area": -1,
            "green_area": -1
        }
        
        try:
            # Получаем объекты в радиусе
            tags_config = {
                'water': {"natural": "water"},
                'building': {"building": True},
                'green': {"landuse": "grass", "natural": "wood"}
            }
            
            for obj_type, tags in tags_config.items():
                    try:
                        gdf = ox.features_from_point((lat, lng), tags=tags, dist=radius)
                    except:
                        continue
                    
                    if not gdf.empty:
                        # Вычисляем расстояния до всех объектов этого типа
                        distances = []
                        areas = []
                        for geom in gdf.geometry:
                            if geom.geom_type == 'Point':
                                obj_point = (geom.y, geom.x)
                                dist = geodesic((lat, lng), obj_point).meters
                                distances.append(dist)
                            elif geom.geom_type in ['Polygon', 'MultiPolygon']:
                                # Для полигонов берем расстояние до ближайшей точки границы
                                for x, y in geom.exterior.coords:
                                    obj_point = (y, x)
                                    dist = geodesic((lat, lng), obj_point).meters
                                    distances.append(dist)
                                # Высчитываем площадь полигона
                                area = geom.area * 1e8 # Домножаем для удобства
                                areas.append(area)
                        
                        # Записываем минимальное расстояние
                        if distances:
                            point_data[f'{obj_type}_dist'] = min(distances)
                            point_data[f'{obj_type}_area'] = sum(areas)
                
                        
        except Exception as e:
            print(f"Ошибка для точки {i}: {e}")
        
        results.append(point_data)
    
    return pd.DataFrame(results)

In [10]:
def process_point(point, radius=500):
    """
    Обрабатывает одну точку для вычисления окружения.
    """
    track_id, lng, lat, alt = point
    point_data = {
        'track_id': track_id,
        'latitude': lat,
        'longitude': lng,
        'altitude': alt,
        'water_dist': -1,
        'building_dist': -1,
        'green_dist': -1,
        "water_area": -1,
        "building_area": -1,
        "green_area": -1
    }

    try:
        # Получаем объекты в радиусе
        tags_config = {
            'water': {"natural": "water"},
            'building': {"building": True},
            'green': {"landuse": "grass", "natural": "wood"}
        }

        for obj_type, tags in tags_config.items():
            try:
                gdf = ox.features_from_point((lat, lng), tags=tags, dist=radius)
            except Exception as e:
                continue

            if not gdf.empty:
                # Вычисляем расстояния до всех объектов этого типа
                distances = []
                areas = []
                for geom in gdf.geometry:
                    if geom.geom_type == 'Point':
                        obj_point = (geom.y, geom.x)
                        dist = geodesic((lat, lng), obj_point).meters
                        distances.append(dist)
                    elif geom.geom_type in ['Polygon', 'MultiPolygon']:
                        # Для полигонов берем расстояние до ближайшей точки границы
                        for x, y in geom.exterior.coords:
                            obj_point = (y, x)
                            dist = geodesic((lat, lng), obj_point).meters
                            distances.append(dist)
                        # Высчитываем площадь полигона
                        area = geom.area * 1e8 # Домножаем для удобства
                        areas.append(area)

                # Записываем минимальное расстояние
                if distances:
                    point_data[f'{obj_type}_dist'] = min(distances)
                # Записываем суммарную площадь
                if areas:
                    point_data[f'{obj_type}_area'] = sum(areas)


    except Exception as e:
        print(f"Ошибка для точки ({lat}, {lng}): {e}")

    return point_data

In [25]:
def multiprocess_execute(points, radius=500, num_threads=4):
    results = []
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = [executor.submit(process_point, point, radius) for point in points]

        # Собираем результаты по мере завершения задач
        for i, future in tqdm(enumerate(as_completed(futures)), total=len(points)):
            try:
                result = future.result()  # Получаем результат из потока
                results.append(result)
                if (i + 1) % 100 == 0:
                    results_df = pd.DataFrame(results)
                    results_df.to_csv("points_data/results.csv")
            except Exception as e:
                print(f"Ошибка при получении результата из потока: {e}")

    return pd.DataFrame(results)

In [12]:
cols = ["track_id", "longitude", "latitude", "altitude"]

points = data.groupby(cols).count().reset_index()[cols]

points

,track_id,longitude,latitude,altitude
0,1,77.205674,28.525768,235.212418
1,1,77.205674,28.525780,235.187759
2,1,77.205674,28.525792,235.160965
3,1,77.205674,28.525804,235.131699
4,1,77.205674,28.525816,235.099670
...,...,...,...,...
1682,3,77.251454,28.672586,211.812775
1683,3,77.251454,28.672592,211.846344
1684,3,77.251456,28.672588,211.823227
1685,3,77.251456,28.672589,211.832489


In [27]:
points_data = multiprocess_execute(points.values.tolist()[1200:1400], num_threads=8)

100%|██████████| 200/200 [06:34<00:00,  1.97s/it]


In [14]:
points_data

,track_id,latitude,longitude,altitude,water_dist,building_dist,green_dist,water_area,building_area,green_area
0,2.0,28.519196,77.210631,233.451874,-1,0.810145,88.677818,-1,1613.216339,3102.917350
1,2.0,28.519183,77.210617,233.195557,-1,2.108750,86.695118,-1,1613.301145,3102.917350
2,2.0,28.519202,77.210637,233.579956,-1,1.252699,89.559521,-1,1611.722966,3102.917350
3,2.0,28.519189,77.210624,233.323563,-1,1.257353,87.649304,-1,1612.593503,3102.917350
4,3.0,28.672769,77.249634,211.539078,-1,18.668746,86.005356,-1,664.658093,104.558847
...,...,...,...,...,...,...,...,...,...,...
95,3.0,28.672494,77.250436,211.701050,-1,7.493336,87.853427,-1,577.498974,104.558847
96,3.0,28.672495,77.250447,211.695770,-1,8.573555,88.763504,-1,577.498974,104.558847
97,3.0,28.672495,77.250457,211.690613,-1,9.548859,89.532132,-1,577.498974,104.558847
98,3.0,28.672495,77.250467,211.685669,-1,10.524560,90.304797,-1,577.498974,104.558847
